# 面试题：RoPE 如何从零实现，为什么它能把相对位置写进 QK 点积？

## 面试回答主线

RoPE 把 Q、K 的每两个隐藏维看作一个二维向量，并按 token 位置乘以不同频率的旋转矩阵。位置 `m` 的 Q 与位置 `n` 的 K 做点积时，两个绝对旋转相消为角度差，因此分数只依赖相对距离 `m-n` 和原始内容向量。它不旋转 V，因为位置信息通过 attention score 决定如何汇聚 V。实现必须保证 `head_dim` 为偶数、Q/K 使用相同频率和 position id，并在 KV cache 解码时从缓存长度继续编号。RoPE 并不自动保证无限长度外推，长上下文还涉及频率缩放与训练分布。

## 真实案例：长客服记录中定位最近一次同主题证据

同一工单中“退款”可能被多次提及，回复时通常更需要靠近当前问题的最近状态。六个案例给出查询位置、三个同内容候选位置和人工目标；候选顺序被打乱，避免固定选第一项蒙对。

In [1]:
import math  # 导入平方根以缩放 QK 点积分数。
import torch  # 导入 PyTorch 以手写旋转位置编码并执行真实梯度训练。
from torch import nn  # 导入基础模块和可学习参数抽象。
import torch.nn.functional as F  # 导入多类交叉熵训练相对位置检索器。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以保证快速复现。
cases = [  # 构造查询位置、候选证据位置和最近证据目标。
    {"ticket": "TK-201", "query_pos": 8, "candidate_pos": [3, 7, 5], "gold": 1},  # 最近候选位于打乱列表的索引一。
    {"ticket": "TK-202", "query_pos": 14, "candidate_pos": [9, 11, 13], "gold": 2},  # 最近候选位于索引二。
    {"ticket": "TK-203", "query_pos": 20, "candidate_pos": [19, 15, 17], "gold": 0},  # 最近候选位于索引零。
    {"ticket": "TK-204", "query_pos": 26, "candidate_pos": [21, 25, 23], "gold": 1},  # 平移后保持相同的相对距离集合。
    {"ticket": "TK-205", "query_pos": 32, "candidate_pos": [29, 27, 31], "gold": 2},  # 更长位置仍要求选择距离一的证据。
    {"ticket": "TK-206", "query_pos": 38, "candidate_pos": [37, 35, 33], "gold": 0},  # 最近候选再次位于索引零。
]  # 结束长客服记录案例列表。
print("工单     query位置  候选位置       人工最近证据")  # 输出真实案例输入表标题。
for item in cases:  # 逐条展示查询位置、候选位置和人工标签。
    gold_position = item["candidate_pos"][item["gold"]]  # 根据候选索引恢复可读的目标位置。
    print(f"{item['ticket']:<8} {item['query_pos']:>9}  {str(item['candidate_pos']):<14} pos={gold_position}")  # 输出一条完整位置检索记录。

工单     query位置  候选位置       人工最近证据
TK-201           8  [3, 7, 5]      pos=7
TK-202          14  [9, 11, 13]    pos=13
TK-203          20  [19, 15, 17]   pos=19
TK-204          26  [21, 25, 23]   pos=25
TK-205          32  [29, 27, 31]   pos=31
TK-206          38  [37, 35, 33]   pos=37


## Baseline（基线）：没有位置时同内容 key 完全并列

三个候选都表示同一主题，若 Q/K 只有内容向量，三个点积分数相同。`argmax` 固定返回索引零，因此候选顺序变化后只能偶尔命中最近证据。

In [2]:
baseline_scores = torch.zeros(len(cases), 3)  # 用相同零差分数表示内容完全相同的三个候选。
baseline_predictions = baseline_scores.argmax(dim=1)  # 使用常见 argmax tie-break 固定选择第零个候选。
gold_indices = torch.tensor([item["gold"] for item in cases], dtype=torch.long)  # 把人工最近证据索引转换为监督张量。
baseline_accuracy = float((baseline_predictions == gold_indices).to(torch.float32).mean())  # 计算无位置方案的检索准确率。
print("工单     baseline分数       预测索引  正确索引")  # 输出基线逐案例结果表标题。
for index, item in enumerate(cases):  # 遍历每个位置检索案例。
    print(f"{item['ticket']:<8} {baseline_scores[index].tolist()} {int(baseline_predictions[index]):>8} {item['gold']:>8}")  # 输出并列分数与固定预测。
print(f"无位置内容匹配准确率：{baseline_accuracy:.1%}")  # 输出后续 RoPE 模型的同数据基线。

工单     baseline分数       预测索引  正确索引
TK-201   [0.0, 0.0, 0.0]        0        1
TK-202   [0.0, 0.0, 0.0]        0        2
TK-203   [0.0, 0.0, 0.0]        0        0
TK-204   [0.0, 0.0, 0.0]        0        1
TK-205   [0.0, 0.0, 0.0]        0        2
TK-206   [0.0, 0.0, 0.0]        0        0
无位置内容匹配准确率：33.3%


## 核心实现一：按偶/奇维显式执行二维旋转

频率为 `10000^(-2i/d)`。下面的函数同时支持任意前导 batch 维，只要求最后一维为偶数；它先计算每个位置的 cos/sin，再对偶数维和奇数维应用旋转公式。

In [3]:
def rotary_frequencies(head_dim):  # 计算每个二维维度对对应的 RoPE 逆频率。
    if head_dim % 2 != 0:  # 每个二维旋转必须同时拥有偶数维和奇数维。
        raise ValueError("RoPE 的 head_dim 必须是偶数")  # 对不可配对维度给出明确配置错误。
    dimension_indices = torch.arange(0, head_dim, 2, dtype=torch.float32)  # 取得每个二维对的偶数维索引。
    return torch.exp(-math.log(10000.0) * dimension_indices / head_dim)  # 返回从高频到低频的几何频率。
def apply_rope(vectors, positions):  # 对最后一维向量按给定位置执行 RoPE 旋转。
    head_dim = vectors.shape[-1]  # 读取需要旋转的单头隐藏维度。
    inverse_frequencies = rotary_frequencies(head_dim).to(vectors.device)  # 在相同设备创建频率向量。
    angles = positions.to(torch.float32).unsqueeze(-1) * inverse_frequencies  # 计算每个位置和频率对应的旋转角。
    cosine = torch.cos(angles)  # 计算所有二维对共享的余弦系数。
    sine = torch.sin(angles)  # 计算所有二维对共享的正弦系数。
    even = vectors[..., 0::2]  # 取得每个二维向量的第一分量。
    odd = vectors[..., 1::2]  # 取得每个二维向量的第二分量。
    rotated = torch.empty_like(vectors)  # 创建与输入同形状的旋转结果。
    rotated[..., 0::2] = even * cosine - odd * sine  # 按二维旋转矩阵计算偶数维输出。
    rotated[..., 1::2] = even * sine + odd * cosine  # 按二维旋转矩阵计算奇数维输出。
    return rotated  # 返回保持范数的旋转后向量。
preview_vector = torch.tensor([[1.0, 0.5, -0.3, 0.8, 0.2, -0.6, 0.9, 0.1]])  # 构造可复核的八维内容向量。
preview_positions = torch.tensor([0])  # 设置第一个预览位置为零。
rotated_zero = apply_rope(preview_vector, preview_positions)  # 旋转位置零应保持原向量不变。
rotated_three = apply_rope(preview_vector, torch.tensor([3]))  # 对相同内容应用位置三的不同角度。
print("RoPE 逆频率：", [round(float(value), 6) for value in rotary_frequencies(8)])  # 输出四个二维对的真实频率。
print("位置0旋转：", [round(float(value), 4) for value in rotated_zero[0]])  # 展示角度零时的恒等结果。
print("位置3旋转：", [round(float(value), 4) for value in rotated_three[0]])  # 展示不同频率二维旋转后的数值。
print("范数对照：", round(float(preview_vector.norm()), 6), round(float(rotated_three.norm()), 6))  # 验证正交旋转保持向量长度。

RoPE 逆频率： [1.0, 0.1, 0.01, 0.001]
位置0旋转： [1.0, 0.5, -0.3, 0.8, 0.2, -0.6, 0.9, 0.1]
位置3旋转： [-1.0606, -0.3539, -0.523, 0.6756, 0.2179, -0.5937, 0.8997, 0.1027]
范数对照： 1.788854 1.788854


## 核心实现二：可学习 Q/K + RoPE 相对位置检索器

模型只有两个可学习内容向量。无位置时所有候选仍然严格并列；开启 RoPE 后，相同 Q/K 在不同相对距离下产生不同点积。训练循环真实执行 `forward → cross_entropy → backward → 手动 SGD`。

In [4]:
query_positions = torch.tensor([item["query_pos"] for item in cases], dtype=torch.long)  # 把六个查询绝对位置整理为批量张量。
candidate_positions = torch.tensor([item["candidate_pos"] for item in cases], dtype=torch.long)  # 把每组三个候选位置整理为二维张量。
class RoPERelativeRetriever(nn.Module):  # 定义用旋转 Q/K 进行相对位置检索的最小网络。
    def __init__(self, head_dim):  # 初始化共享给全部案例的内容查询和内容键。
        super().__init__()  # 注册基础模块状态以追踪参数。
        self.head_dim = head_dim  # 保存单头隐藏维度供缩放使用。
        self.query_vector = nn.Parameter(torch.randn(head_dim) * 0.20)  # 创建可学习主题查询内容向量。
        self.key_vector = nn.Parameter(torch.randn(head_dim) * 0.20)  # 创建可学习同主题证据内容向量。
    def forward(self, query_pos, key_pos, use_rope=True):  # 计算每个查询对三个候选证据的分数。
        batch_size, candidate_count = key_pos.shape  # 读取案例数和每案候选数。
        queries = self.query_vector.reshape(1, 1, -1).expand(batch_size, candidate_count, -1)  # 把共享 Q 扩展到每个查询候选 pair。
        keys = self.key_vector.reshape(1, 1, -1).expand(batch_size, candidate_count, -1)  # 把共享 K 扩展到每个查询候选 pair。
        expanded_query_pos = query_pos.unsqueeze(1).expand_as(key_pos)  # 把每个查询位置复制到对应三个候选。
        if use_rope:  # 正常路径需要把绝对位置旋转进 Q 和 K。
            queries = apply_rope(queries, expanded_query_pos)  # 按查询绝对位置旋转 Q。
            keys = apply_rope(keys, key_pos)  # 按候选证据绝对位置旋转 K。
        scores = (queries * keys).sum(dim=-1) / math.sqrt(self.head_dim)  # 计算缩放后的逐候选 QK 点积。
        return scores  # 返回可直接用于检索或交叉熵的分数矩阵。
torch.manual_seed(71)  # 固定相对位置检索器的参数初始化。
rope_model = RoPERelativeRetriever(8)  # 创建八维 RoPE 检索器。
training_trace = []  # 保存代表轮次的损失、准确率和梯度范数。
for step in range(501):  # 执行真实的前向、反向和手动参数更新。
    scores = rope_model(query_positions, candidate_positions, use_rope=True)  # 计算六案三候选的相对位置分数。
    loss = F.cross_entropy(scores, gold_indices)  # 用人工最近证据索引计算多类交叉熵。
    loss.backward()  # 对共享 Q/K 内容向量执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum()) for parameter in rope_model.parameters()))  # 计算本轮所有参数的全局梯度范数。
    with torch.no_grad():  # 参数更新不应构建新的计算图。
        for parameter in rope_model.parameters():  # 遍历可学习查询和键向量。
            parameter -= 0.12 * parameter.grad  # 使用固定学习率执行手动 SGD。
            parameter.grad.zero_()  # 清空梯度避免跨轮错误累积。
    if step in {0, 1, 10, 50, 150, 500}:  # 记录能够说明真实优化过程的代表轮次。
        accuracy = float((scores.detach().argmax(dim=1) == gold_indices).to(torch.float32).mean())  # 计算本轮最近证据准确率。
        training_trace.append((step, float(loss.detach()), gradient_norm, accuracy))  # 保存轮次、损失、梯度和准确率。
print("轮次 | 交叉熵  | 梯度范数 | 准确率")  # 输出训练轨迹表标题。
for step, loss_value, gradient_norm, accuracy in training_trace:  # 逐个展示代表训练状态。
    print(f"{step:>4} | {loss_value:>7.4f} | {gradient_norm:>8.4f} | {accuracy:>6.1%}")  # 输出真实损失、梯度和准确率。

轮次 | 交叉熵  | 梯度范数 | 准确率
   0 |  1.0690 |   0.1648 | 100.0%
   1 |  1.0657 |   0.1695 | 100.0%
  10 |  1.0245 |   0.2274 | 100.0%
  50 |  0.3487 |   0.3819 | 100.0%
 150 |  0.0225 |   0.0488 | 100.0%
 500 |  0.0033 |   0.0087 | 100.0%


## 结果表与相对位移不变量

逐案打印相对距离、分数和预测；再把同一 Q/K 的两个位置同时平移 17，检查点积保持不变。绝对位置可以很大，核心关系仍由差值决定。

In [5]:
with torch.no_grad():  # 评估阶段关闭梯度图以得到确定性分数。
    rope_scores = rope_model(query_positions, candidate_positions, use_rope=True)  # 计算最终 RoPE 候选分数。
    rope_predictions = rope_scores.argmax(dim=1)  # 选择每案得分最高的证据索引。
rope_accuracy = float((rope_predictions == gold_indices).to(torch.float32).mean())  # 计算 RoPE 检索准确率。
print("工单     相对距离       RoPE分数                 预测/正确")  # 输出逐案例检索结果表标题。
for index, item in enumerate(cases):  # 遍历六个真实位置案例。
    distances = [item["query_pos"] - position for position in item["candidate_pos"]]  # 计算查询到每个历史证据的相对距离。
    score_view = [round(float(value), 4) for value in rope_scores[index]]  # 格式化三个学习后分数。
    print(f"{item['ticket']:<8} {str(distances):<14} {str(score_view):<25} {int(rope_predictions[index])}/{item['gold']}")  # 输出相对距离、分数和索引对照。
base_q_position = torch.tensor([12])  # 选择一组绝对查询位置用于平移测试。
base_k_positions = torch.tensor([[11, 9, 7]])  # 选择相对距离一、三、五的候选位置。
shifted_q_position = base_q_position + 17  # 同时把查询绝对位置向后平移十七。
shifted_k_positions = base_k_positions + 17  # 同时把全部键绝对位置平移相同距离。
with torch.no_grad():  # 平移不变量检查不需要梯度图。
    base_shift_scores = rope_model(base_q_position, base_k_positions, use_rope=True)  # 计算平移前的三个相对分数。
    shifted_scores = rope_model(shifted_q_position, shifted_k_positions, use_rope=True)  # 计算平移后的三个相对分数。
shift_difference = float((base_shift_scores - shifted_scores).abs().max())  # 量化共同平移前后的最大数值差异。
print(f"准确率对照：无位置={baseline_accuracy:.1%}，RoPE={rope_accuracy:.1%}")  # 输出同数据上的核心检索对照。
print("共同平移17前的分数：", [round(float(value), 6) for value in base_shift_scores[0]])  # 输出平移前分数。
print("共同平移17后的分数：", [round(float(value), 6) for value in shifted_scores[0]], f"，最大差={shift_difference:.8f}")  # 输出平移后分数和误差。

工单     相对距离       RoPE分数                 预测/正确
TK-201   [5, 1, 3]      [-2.4942, 4.0635, -2.1973] 1/1
TK-202   [5, 3, 1]      [-2.4942, -2.1973, 4.0635] 2/2
TK-203   [1, 5, 3]      [4.0635, -2.4942, -2.1973] 0/0
TK-204   [5, 1, 3]      [-2.4942, 4.0635, -2.1973] 1/1
TK-205   [3, 5, 1]      [-2.1973, -2.4942, 4.0635] 2/2
TK-206   [1, 3, 5]      [4.0635, -2.1973, -2.4942] 0/0
准确率对照：无位置=33.3%，RoPE=100.0%
共同平移17前的分数： [4.063466, -2.197342, -2.494195]
共同平移17后的分数： [4.063466, -2.197342, -2.494195] ，最大差=0.00000024


## 结果解读

无位置模型面对相同内容只能打平，RoPE 让共享 Q/K 的点积随相对距离变化，并通过真实梯度学会在这批工单中偏好最近证据。共同平移不改变分数，数值上验证了相对位置性质；但“最近”是否正确仍由任务监督决定，RoPE 本身不会自动理解业务时间。

## 失败案例：KV cache 解码把新 token 的 position 重置为 0

预填充长度为 5 时，下一个 token 的绝对位置应为 5。若增量解码每步都传位置 0，旋转后的 Q 与全量前向不一致；使用 `cache_length` 作为 offset 后可逐位对齐。

In [6]:
cached_length = 5  # 模拟已经预填充五个 token 的 KV cache。
decode_query = rope_model.query_vector.detach().reshape(1, -1)  # 取得当前模型学习后的新 token 查询内容向量。
full_forward_query = apply_rope(decode_query, torch.tensor([cached_length]))  # 用全量前向应有的绝对位置五计算参考旋转 Q。
wrong_cached_query = apply_rope(decode_query, torch.tensor([0]))  # 错误地把每个增量 token 位置重置为零。
fixed_cached_query = apply_rope(decode_query, torch.tensor([cached_length]))  # 使用缓存长度作为增量 position offset。
wrong_cache_error = float((full_forward_query - wrong_cached_query).abs().max())  # 量化错误位置导致的 Q 表示漂移。
fixed_cache_error = float((full_forward_query - fixed_cached_query).abs().max())  # 量化正确 offset 与全量前向的差异。
previous_key = rope_model.key_vector.detach().reshape(1, -1)  # 取得缓存中前一 token 的内容键向量。
rotated_previous_key = apply_rope(previous_key, torch.tensor([cached_length - 1]))  # 按真实位置四旋转缓存键。
wrong_score = float((wrong_cached_query * rotated_previous_key).sum() / math.sqrt(rope_model.head_dim))  # 计算错误重置位置后的相对分数。
fixed_score = float((fixed_cached_query * rotated_previous_key).sum() / math.sqrt(rope_model.head_dim))  # 计算正确相邻位置的相对分数。
print(f"错误 position=0：Q 最大漂移={wrong_cache_error:.6f}，对前一 token 分数={wrong_score:.4f}")  # 展示缓存位置重置的实际影响。
print(f"修复 position=cache_length：Q 最大漂移={fixed_cache_error:.6f}，对前一 token 分数={fixed_score:.4f}")  # 展示 offset 修复与全量前向一致。

错误 position=0：Q 最大漂移=4.094283，对前一 token 分数=0.7412
修复 position=cache_length：Q 最大漂移=0.000000，对前一 token 分数=4.0635


## 生产差距与落地清单

教学模型只有单头八维位置检索。真实 LLM 要在 `[batch, heads, length, head_dim]` 上广播 cos/sin，并处理张量并行、GQA、KV cache 分页、左 padding 和连续批处理 position id。长上下文扩展需要选择 NTK/linear/YaRN 等缩放策略并回放困惑度与 needle 任务，不能只看旋转公式可运行。缓存键还必须绑定模型版本、RoPE 配置和绝对 offset。

## 最小回归测试

断言只保护旋转范数、相对位移与 cache offset；频率表、真实训练轨迹和逐案分数才是主要学习证据。

In [7]:
assert torch.allclose(rotated_zero, preview_vector)  # 验证位置零对应恒等旋转。
assert abs(float(preview_vector.norm() - rotated_three.norm())) < 1e-6  # 验证二维正交旋转保持向量范数。
assert training_trace[-1][1] < training_trace[0][1]  # 验证真实反向传播降低相对位置检索交叉熵。
assert rope_accuracy > baseline_accuracy  # 验证 RoPE 相对位置模型优于无位置并列基线。
assert shift_difference < 1e-5  # 验证 Q/K 共同平移后点积分数保持不变。
assert wrong_cache_error > 1e-3  # 固化增量 position 重置导致表示漂移的失败案例。
assert fixed_cache_error < 1e-7  # 验证使用 cache length 后与全量前向严格一致。
print("最小回归测试通过：旋转公式、相对位移检索和 KV cache 位置续接均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：旋转公式、相对位移检索和 KV cache 位置续接均符合预期。
